# Diversified Multi-Clip Video Dataset

This notebook builds a diversified video index under `datasets/diversified` and defines PyTorch Dataset/DataLoader utilities for weakly labeled video classification.

Design:
- One video yields multiple independent clips per epoch.
- Training clips use segment coverage, variable clip length, random stride, and non-overlap constraints.
- Class-balanced sampling operates at the video level and is projected to clip samples.
- Validation/test use deterministic multi-clip inference and average logits across clips.


In [15]:
from pathlib import Path
from dataclasses import dataclass
import json
import math
import random
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode


PROJECT_ROOT = Path(r"E:/m-hvc")
SOURCE_DATASET_DIR = PROJECT_ROOT / "datasets" / "old-dataset"
MANIFEST_PATH = SOURCE_DATASET_DIR / "video_df.csv"
FRAMES_ROOT = SOURCE_DATASET_DIR / "frames_unique"
OUT_DIR = PROJECT_ROOT / "datasets" / "diversified"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "seed": 42,
    "test_size": 0.15,
    "val_size": 0.15,
    "clips_per_video_train": 3,  # use 2 or 3
    "eval_clips_per_video": 5,   # use 3-5
    "clip_lengths": [16, 24, 32],
    "strides": [1, 2, 3, 4],
    "eval_stride": 2,
    "min_start_gap_fraction": 0.25,
    "resample_attempts": 32,
    "resize_short_side": 256,
    "crop_size": 224,
    "horizontal_flip": True,
    "tensor_layout": "CTHW",  # CTHW or THWC
    "offline_min_clips": 2,
    "offline_max_clips": 4,
    "motion_filter": False,
    "motion_top_fraction": 0.8,
    "low_motion_threshold": 1e-4,
    "normalization_mean": [0.485, 0.456, 0.406],
    "normalization_std": [0.229, 0.224, 0.225],
    "clips_dir": str(OUT_DIR / "clips"),
    "materialized_clip_index": str(OUT_DIR / "materialized_clip_index.csv"),
    "materialized_dtype": "uint8",
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(CFG["seed"])


C:\Users\Rasheek\.conda\envs\fyp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Build Video-Level Index


In [16]:
def numeric_frame_key(path: Path):
    stem = path.stem.split("_", 1)[0]
    return (0, int(stem)) if stem.isdigit() else (1, path.stem)


def list_frame_paths(frame_dir):
    frame_dir = Path(frame_dir)
    if not frame_dir.exists():
        return []
    paths = [path for path in frame_dir.iterdir() if path.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    return sorted(paths, key=numeric_frame_key)


manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df["source_video_id"] = manifest_df["source_video_id"].astype(str)
manifest_df["label"] = manifest_df["label"].astype(int)
manifest_df["frame_dir"] = manifest_df["source_video_id"].map(lambda value: str(FRAMES_ROOT / str(value)))
manifest_df["num_frames"] = manifest_df["frame_dir"].map(lambda value: len(list_frame_paths(value)))
manifest_df = manifest_df[(manifest_df["label"].isin([0, 1])) & (manifest_df["num_frames"] > 0)].copy()
manifest_df = manifest_df.reset_index(drop=True)

assert manifest_df["source_video_id"].is_unique

video_index_path = OUT_DIR / "video_index.csv"
manifest_df.to_csv(video_index_path, index=False)
print(f"saved {len(manifest_df):,} indexed videos -> {video_index_path}")
display(manifest_df[["label", "num_frames"]].groupby("label").agg(["count", "mean", "median"]))


saved 3,281 indexed videos -> E:\m-hvc\datasets\diversified\video_index.csv


num_frames                  
           count       mean median
label                             
0           1948  47.331109   64.0
1           1333  45.308327   64.0

## Stratified Splits


In [17]:
train_val_df, test_df = train_test_split(
    manifest_df,
    test_size=CFG["test_size"],
    random_state=CFG["seed"],
    stratify=manifest_df["label"],
)
relative_val_size = CFG["val_size"] / (1.0 - CFG["test_size"])
train_df, val_df = train_test_split(
    train_val_df,
    test_size=relative_val_size,
    random_state=CFG["seed"],
    stratify=train_val_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

splits_df = pd.concat(
    [train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")],
    ignore_index=True,
)
splits_path = OUT_DIR / "splits.csv"
splits_df.to_csv(splits_path, index=False)
print(f"saved splits -> {splits_path}")
display(pd.crosstab(splits_df["split"], splits_df["label"]))


saved splits -> E:\m-hvc\datasets\diversified\splits.csv


label,0,1
split,,
test,293,200
train,1362,933
val,293,200


## Clip Sampling Rules


In [18]:
SEGMENTS = [(0.0, 1.0 / 3.0), (1.0 / 3.0, 2.0 / 3.0), (2.0 / 3.0, 1.0)]


@dataclass(frozen=True)
class ClipSpec:
    video_id: str
    label: int
    start: int
    stride: int
    length: int
    segment: int
    num_frames: int

    @property
    def end(self):
        return self.start + (self.length - 1) * self.stride


def segment_bounds(num_frames, segment_id):
    lo_frac, hi_frac = SEGMENTS[int(segment_id)]
    lo = int(math.floor(num_frames * lo_frac))
    hi = max(lo, int(math.ceil(num_frames * hi_frac)) - 1)
    return min(lo, max(0, num_frames - 1)), min(hi, max(0, num_frames - 1))


def window_indices(start, length, stride, num_frames):
    if num_frames <= 0:
        raise ValueError("num_frames must be positive")
    indices = start + np.arange(length) * stride
    indices = np.clip(indices, 0, num_frames - 1)
    if len(indices) < length:
        indices = np.pad(indices, (0, length - len(indices)), constant_values=num_frames - 1)
    return indices.astype(np.int64)


def starts_far_enough(candidate_start, chosen_specs, num_frames, gap_fraction):
    min_gap = max(1, int(round(num_frames * gap_fraction)))
    return all(abs(candidate_start - spec.start) >= min_gap for spec in chosen_specs)


def windows_overlap(a_start, a_length, a_stride, b_spec):
    a = set(window_indices(a_start, a_length, a_stride, b_spec.num_frames).tolist())
    b = set(window_indices(b_spec.start, b_spec.length, b_spec.stride, b_spec.num_frames).tolist())
    return bool(a & b)


def violates_diversity(candidate_start, candidate_length, candidate_stride, chosen_specs, num_frames, gap_fraction):
    if not starts_far_enough(candidate_start, chosen_specs, num_frames, gap_fraction):
        return True
    for spec in chosen_specs:
        if candidate_stride == spec.stride and windows_overlap(candidate_start, candidate_length, candidate_stride, spec):
            return True
    return False


def frame_motion_scores(frame_paths):
    scores = np.zeros(len(frame_paths), dtype=np.float32)
    previous = None
    for idx, frame_path in enumerate(frame_paths):
        with Image.open(frame_path) as image:
            gray = np.asarray(image.convert("L").resize((64, 64)), dtype=np.float32) / 255.0
        if previous is not None:
            scores[idx] = float(np.mean(np.abs(gray - previous)))
        previous = gray
    return scores


def prefer_motion_rich_start(frame_dir, candidate_starts, length, stride, rng, cfg=CFG):
    if not cfg.get("motion_filter", False) or not candidate_starts:
        return int(rng.choice(candidate_starts))
    frame_paths = list_frame_paths(frame_dir)
    if len(frame_paths) <= 1:
        return int(rng.choice(candidate_starts))
    scores = frame_motion_scores(frame_paths)
    window_scores = []
    for start in candidate_starts:
        indices = window_indices(start, length, stride, len(frame_paths))
        window_scores.append(float(np.mean(scores[indices])))
    window_scores = np.asarray(window_scores, dtype=np.float32)
    if float(window_scores.max(initial=0.0)) < float(cfg.get("low_motion_threshold", 0.0)):
        return int(rng.choice(candidate_starts))
    cutoff = np.quantile(window_scores, 1.0 - float(cfg.get("motion_top_fraction", 0.8)))
    preferred = [start for start, score in zip(candidate_starts, window_scores) if score >= cutoff]
    return int(rng.choice(preferred or candidate_starts))


def sample_clip_spec(row, segment_id, rng, chosen_specs=None, cfg=CFG):
    chosen_specs = chosen_specs or []
    num_frames = int(row.num_frames)
    length = int(rng.choice(cfg["clip_lengths"]))
    gap_fraction = float(cfg["min_start_gap_fraction"])
    seg_lo, seg_hi = segment_bounds(num_frames, segment_id)

    best_spec = None
    for _ in range(int(cfg["resample_attempts"])):
        stride = int(rng.choice(cfg["strides"]))
        max_start_for_stride = max(0, num_frames - 1)
        lo = min(seg_lo, max_start_for_stride)
        hi = min(seg_hi, max_start_for_stride)
        if hi < lo:
            lo, hi = 0, max_start_for_stride
        candidate_starts = list(range(lo, hi + 1)) if hi > lo else [int(lo)]
        start = prefer_motion_rich_start(row.frame_dir, candidate_starts, length, stride, rng, cfg)
        spec = ClipSpec(str(row.source_video_id), int(row.label), start, stride, length, int(segment_id), num_frames)
        best_spec = spec
        if not violates_diversity(start, length, stride, chosen_specs, num_frames, gap_fraction):
            return spec
    return best_spec


def sample_specs_for_video(row, clips_per_video, rng, cfg=CFG):
    clips_per_video = int(clips_per_video)
    if clips_per_video == 3:
        segment_ids = [0, 1, 2]
    elif clips_per_video == 2:
        segment_ids = rng.choice([0, 1, 2], size=2, replace=False).tolist()
    else:
        segment_ids = rng.choice([0, 1, 2], size=clips_per_video, replace=True).tolist()

    specs = []
    for segment_id in segment_ids:
        specs.append(sample_clip_spec(row, segment_id, rng, specs, cfg))
    return specs


def deterministic_eval_specs(row, clips_per_video=5, cfg=CFG):
    num_frames = int(row.num_frames)
    length = max(int(x) for x in cfg["clip_lengths"])
    stride = int(cfg["eval_stride"])
    starts = np.linspace(0, max(0, num_frames - 1), num=max(1, int(clips_per_video)))
    specs = []
    for view_idx, start in enumerate(starts):
        segment_id = min(2, int(view_idx * 3 / max(1, int(clips_per_video))))
        specs.append(
            ClipSpec(
                video_id=str(row.source_video_id),
                label=int(row.label),
                start=int(round(start)),
                stride=stride,
                length=length,
                segment=segment_id,
                num_frames=num_frames,
            )
        )
    return specs


## Optional Offline Clip Expansion


In [19]:
def build_offline_clip_index(video_df, output_path=OUT_DIR / "offline_clip_index.csv", cfg=CFG):
    rng = np.random.default_rng(int(cfg["seed"]))
    rows = []
    for row in video_df.itertuples(index=False):
        num_clips = int(rng.integers(int(cfg["offline_min_clips"]), int(cfg["offline_max_clips"]) + 1))
        specs = sample_specs_for_video(row, num_clips, rng, cfg)
        for clip_idx, spec in enumerate(specs):
            rows.append(
                {
                    "source_video_id": spec.video_id,
                    "label": spec.label,
                    "clip_idx": clip_idx,
                    "start": spec.start,
                    "stride": spec.stride,
                    "length": spec.length,
                    "segment": spec.segment,
                    "num_frames": spec.num_frames,
                }
            )
    clip_index_df = pd.DataFrame(rows)
    clip_index_df.to_csv(output_path, index=False)
    return clip_index_df


offline_clip_index_df = build_offline_clip_index(manifest_df)
print(f"saved offline clip specs -> {OUT_DIR / 'offline_clip_index.csv'}")
display(offline_clip_index_df.head())


saved offline clip specs -> E:\m-hvc\datasets\diversified\offline_clip_index.csv


,source_video_id,label,clip_idx,start,stride,length,segment,num_frames
0,R_hate_video_099,1,0,43,4,24,2,64
1,R_hate_video_099,1,1,23,1,32,1,64
2,R_hate_video_100,1,0,16,3,32,0,64
3,R_hate_video_100,1,1,32,4,32,1,64
4,R_hate_video_100,1,2,62,1,16,2,64


## Spatial Preprocessing And Clip Loading


In [20]:
def resize_short_side(image, short_side):
    width, height = image.size
    if min(width, height) == short_side:
        return image
    scale = short_side / min(width, height)
    new_width = int(round(width * scale))
    new_height = int(round(height * scale))
    return TF.resize(image, [new_height, new_width], interpolation=InterpolationMode.BILINEAR)


def crop_params(image, crop_size, train, rng):
    width, height = image.size
    if train:
        top = int(rng.integers(0, max(1, height - crop_size + 1)))
        left = int(rng.integers(0, max(1, width - crop_size + 1)))
    else:
        top = max(0, (height - crop_size) // 2)
        left = max(0, (width - crop_size) // 2)
    return top, left, min(crop_size, height), min(crop_size, width)


def load_clip_from_frames(frame_dir, spec, train, rng, cfg=CFG):
    frame_paths = list_frame_paths(frame_dir)
    if not frame_paths:
        raise FileNotFoundError(f"No frames found in {frame_dir}")
    indices = window_indices(spec.start, spec.length, spec.stride, len(frame_paths))
    do_flip = bool(train and cfg["horizontal_flip"] and rng.random() < 0.5)

    first = resize_short_side(Image.open(frame_paths[int(indices[0])]).convert("RGB"), int(cfg["resize_short_side"]))
    top, left, crop_h, crop_w = crop_params(first, int(cfg["crop_size"]), train, rng)
    first.close()

    tensors = []
    for index in indices:
        with Image.open(frame_paths[int(index)]) as image:
            image = image.convert("RGB")
            image = resize_short_side(image, int(cfg["resize_short_side"]))
            image = TF.crop(image, top, left, crop_h, crop_w)
            image = TF.resize(image, [int(cfg["crop_size"]), int(cfg["crop_size"])], interpolation=InterpolationMode.BILINEAR)
            if do_flip:
                image = ImageOps.mirror(image)
            tensor = TF.to_tensor(image)
            tensor = TF.normalize(tensor, cfg["normalization_mean"], cfg["normalization_std"])
            tensors.append(tensor)

    clip = torch.stack(tensors, dim=1)  # [C,T,H,W]
    if cfg["tensor_layout"].upper() == "THWC":
        clip = clip.permute(1, 2, 3, 0).contiguous()
    return clip


def pad_collate_clips(batch):
    clips, labels, video_ids, specs = zip(*batch)
    layout = CFG["tensor_layout"].upper()
    max_t = max(clip.shape[1] if layout == "CTHW" else clip.shape[0] for clip in clips)
    padded = []
    for clip in clips:
        if layout == "CTHW":
            pad_t = max_t - clip.shape[1]
            if pad_t > 0:
                clip = F.pad(clip, (0, 0, 0, 0, 0, pad_t))
        else:
            pad_t = max_t - clip.shape[0]
            if pad_t > 0:
                clip = F.pad(clip, (0, 0, 0, 0, 0, 0, 0, pad_t))
        padded.append(clip)
    return {
        "clip": torch.stack(padded, dim=0),
        "label": torch.tensor(labels, dtype=torch.float32),
        "video_id": list(video_ids),
        "spec": list(specs),
        "length": torch.tensor([spec.length for spec in specs], dtype=torch.long),
    }


## Materialize Diversified Clips To Disk

The earlier cells create an index of temporal clip specs. This section actually preprocesses those clips and stores tensors under `datasets/diversified/clips`.

Storage note: full materialization can be large. By default the example call below uses a small smoke limit. Set `max_clips=None` in `materialize_clip_tensors(...)` to write the full diversified dataset.


In [27]:
def load_clip_uint8_from_frames(frame_dir, spec, cfg=CFG):
    """Deterministically preprocess a clip and return uint8 [C,T,H,W].

    Temporal diversity is fixed by ClipSpec. Spatial preprocessing is deterministic:
    resize shorter side, center crop, and store uint8 values to keep disk usage lower.
    Normalization remains in the Dataset so train/val/test use identical stats.
    """
    frame_paths = list_frame_paths(frame_dir)
    if not frame_paths:
        raise FileNotFoundError(f"No frames found in {frame_dir}")
    indices = window_indices(spec.start, spec.length, spec.stride, len(frame_paths))

    first = resize_short_side(Image.open(frame_paths[int(indices[0])]).convert("RGB"), int(cfg["resize_short_side"]))
    top, left, crop_h, crop_w = crop_params(first, int(cfg["crop_size"]), train=False, rng=np.random.default_rng(0))
    first.close()

    frames = []
    for index in indices:
        with Image.open(frame_paths[int(index)]) as image:
            image = image.convert("RGB")
            image = resize_short_side(image, int(cfg["resize_short_side"]))
            image = TF.crop(image, top, left, crop_h, crop_w)
            image = TF.resize(image, [int(cfg["crop_size"]), int(cfg["crop_size"])], interpolation=InterpolationMode.BILINEAR)
            arr = np.asarray(image, dtype=np.uint8)
            frames.append(torch.from_numpy(arr).permute(2, 0, 1).contiguous())
    return torch.stack(frames, dim=1)  # [C,T,H,W], uint8


def clipspec_to_dict(spec):
    return {
        "video_id": spec.video_id,
        "label": int(spec.label),
        "start": int(spec.start),
        "stride": int(spec.stride),
        "length": int(spec.length),
        "segment": int(spec.segment),
        "num_frames": int(spec.num_frames),
    }


def dict_to_clipspec(row):
    return ClipSpec(
        video_id=str(row.source_video_id),
        label=int(row.label),
        start=int(row.start),
        stride=int(row.stride),
        length=int(row.length),
        segment=int(row.segment),
        num_frames=int(row.num_frames),
    )


def materialize_clip_tensors(
    clip_index_df,
    splits_df,
    output_dir=None,
    output_index_path=None,
    max_clips=None,
    overwrite=False,
    cfg=CFG,
):
    output_dir = Path(output_dir or cfg["clips_dir"])
    output_index_path = Path(output_index_path or cfg["materialized_clip_index"])
    output_dir.mkdir(parents=True, exist_ok=True)

    video_lookup = splits_df[["source_video_id", "split", "frame_dir", "video_path"]].copy()
    video_lookup["source_video_id"] = video_lookup["source_video_id"].astype(str)
    work_df = clip_index_df.copy()
    work_df["source_video_id"] = work_df["source_video_id"].astype(str)
    work_df = work_df.merge(video_lookup, on="source_video_id", how="left")
    if work_df["frame_dir"].isna().any():
        missing = work_df.loc[work_df["frame_dir"].isna(), "source_video_id"].head().tolist()
        raise ValueError(f"Clip index contains videos missing from splits_df: {missing}")
    if max_clips is not None:
        work_df = work_df.head(int(max_clips)).copy()

    rows = []
    for row in tqdm(list(work_df.itertuples(index=False)), desc="materializing clips"):
        spec = dict_to_clipspec(row)
        split_dir = output_dir / str(row.split)
        video_dir = split_dir / str(row.source_video_id)
        video_dir.mkdir(parents=True, exist_ok=True)
        clip_name = f"clip_{int(row.clip_idx):03d}_s{spec.start}_k{spec.stride}_t{spec.length}.pt"
        clip_path = video_dir / clip_name

        if overwrite or not clip_path.exists():
            clip_tensor = load_clip_uint8_from_frames(row.frame_dir, spec, cfg)
            payload = {
                "clip": clip_tensor,
                "label": int(row.label),
                "source_video_id": str(row.source_video_id),
                "split": str(row.split),
                "spec": clipspec_to_dict(spec),
            }
            torch.save(payload, clip_path)

        rows.append(
            {
                "source_video_id": str(row.source_video_id),
                "label": int(row.label),
                "split": str(row.split),
                "clip_idx": int(row.clip_idx),
                "start": spec.start,
                "stride": spec.stride,
                "length": spec.length,
                "segment": spec.segment,
                "num_frames": spec.num_frames,
                "clip_path": str(clip_path),
            }
        )

    materialized_df = pd.DataFrame(rows)
    if output_index_path.exists() and not overwrite and max_clips is not None:
        previous_df = pd.read_csv(output_index_path)
        materialized_df = pd.concat([previous_df, materialized_df], ignore_index=True)
        materialized_df = materialized_df.drop_duplicates("clip_path", keep="last")
    materialized_df.to_csv(output_index_path, index=False)
    return materialized_df


# Smoke materialization: writes a few real preprocessed clip tensors.
# For the full dataset, run:
# materialized_clip_df = materialize_clip_tensors(offline_clip_index_df, splits_df, max_clips=None, overwrite=False)
materialized_clip_df = materialize_clip_tensors(
    offline_clip_index_df,
    splits_df,
    overwrite=False,
)
print(f"materialized clips indexed: {len(materialized_clip_df):,}")
display(materialized_clip_df.head())


materializing clips: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9845/9845 [23:52<00:00,  6.87it/s]

materialized clips indexed: 9,845


,source_video_id,label,split,clip_idx,start,stride,length,segment,num_frames,clip_path
0,R_hate_video_099,1,train,0,43,4,24,2,64,E:\m-hvc\datasets\diversified\clips\train\R_ha...
1,R_hate_video_099,1,train,1,23,1,32,1,64,E:\m-hvc\datasets\diversified\clips\train\R_ha...
2,R_hate_video_100,1,test,0,16,3,32,0,64,E:\m-hvc\datasets\diversified\clips\test\R_hat...
3,R_hate_video_100,1,test,1,32,4,32,1,64,E:\m-hvc\datasets\diversified\clips\test\R_hat...
4,R_hate_video_100,1,test,2,62,1,16,2,64,E:\m-hvc\datasets\diversified\clips\test\R_hat...


## Dataset For Materialized Clips


In [22]:
class MaterializedClipDataset(Dataset):
    def __init__(self, materialized_df, split=None, train=False, cfg=CFG):
        self.df = materialized_df.copy()
        if split is not None:
            self.df = self.df[self.df["split"].astype(str) == str(split)].copy()
        self.df = self.df.reset_index(drop=True)
        self.train = bool(train)
        self.cfg = cfg
        self.mean = torch.tensor(cfg["normalization_mean"], dtype=torch.float32).view(3, 1, 1, 1)
        self.std = torch.tensor(cfg["normalization_std"], dtype=torch.float32).view(3, 1, 1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        payload = torch.load(row.clip_path, map_location="cpu")
        clip = payload["clip"]
        if clip.dtype == torch.uint8:
            clip = clip.float().div(255.0)
        else:
            clip = clip.float()
        clip = (clip - self.mean) / self.std
        if self.train and CFG["horizontal_flip"] and random.random() < 0.5:
            clip = torch.flip(clip, dims=[-1])
        return {
            "clip": clip,
            "label": torch.tensor(float(payload["label"]), dtype=torch.float32),
            "source_video_id": str(payload["source_video_id"]),
            "spec": payload["spec"],
        }


def collate_materialized_clips(batch):
    clips = [item["clip"] for item in batch]
    max_t = max(clip.shape[1] for clip in clips)
    padded = []
    for clip in clips:
        pad_t = max_t - clip.shape[1]
        if pad_t > 0:
            clip = F.pad(clip, (0, 0, 0, 0, 0, pad_t))
        padded.append(clip)
    return {
        "clip": torch.stack(padded, dim=0),
        "label": torch.stack([item["label"] for item in batch]),
        "source_video_id": [item["source_video_id"] for item in batch],
        "spec": [item["spec"] for item in batch],
    }


materialized_dataset = MaterializedClipDataset(materialized_clip_df, train=True)
materialized_loader = DataLoader(materialized_dataset, batch_size=4, shuffle=True, collate_fn=collate_materialized_clips)
materialized_batch = next(iter(materialized_loader))
print("materialized batch:", tuple(materialized_batch["clip"].shape))
print("materialized labels:", materialized_batch["label"].tolist())


materialized batch: (4, 3, 32, 224, 224)
materialized labels: [1.0, 1.0, 1.0, 1.0]


C:\Users\Rasheek\AppData\Local\Temp\ipykernel_4712\207688123.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(row.clip_path, map_location="cpu")


## Dataset: Video-Level Index, Clip-Level Samples


In [23]:
class MultiClipVideoDataset(Dataset):
    def __init__(
        self,
        video_df,
        clips_per_video=3,
        train=True,
        deterministic=False,
        hard_video_weights=None,
        seed=42,
        cfg=CFG,
    ):
        self.video_df = video_df.reset_index(drop=True).copy()
        self.clips_per_video = int(clips_per_video)
        self.train = bool(train)
        self.deterministic = bool(deterministic)
        self.seed = int(seed)
        self.cfg = cfg
        self.epoch = 0
        self.hard_video_weights = hard_video_weights or {}
        self._spec_cache = {}

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self._spec_cache.clear()

    def __len__(self):
        return len(self.video_df) * self.clips_per_video

    def _video_clip_slot(self, index):
        video_idx = int(index) // self.clips_per_video
        clip_slot = int(index) % self.clips_per_video
        return video_idx, clip_slot

    def _rng_for_video(self, video_idx):
        row = self.video_df.iloc[video_idx]
        stable = abs(hash(str(row.source_video_id))) % (2**31)
        return np.random.default_rng(self.seed + self.epoch * 100_003 + stable)

    def _specs_for_video(self, video_idx):
        if video_idx in self._spec_cache:
            return self._spec_cache[video_idx]
        row = self.video_df.iloc[video_idx]
        if self.deterministic:
            specs = deterministic_eval_specs(row, self.clips_per_video, self.cfg)
        else:
            rng = self._rng_for_video(video_idx)
            specs = sample_specs_for_video(row, self.clips_per_video, rng, self.cfg)
        self._spec_cache[video_idx] = specs
        return specs

    def __getitem__(self, index):
        video_idx, clip_slot = self._video_clip_slot(index)
        row = self.video_df.iloc[video_idx]
        spec = self._specs_for_video(video_idx)[clip_slot]
        rng = np.random.default_rng(self.seed + self.epoch * 1_000_003 + int(index))
        clip = load_clip_from_frames(row.frame_dir, spec, train=self.train and not self.deterministic, rng=rng, cfg=self.cfg)
        return clip, int(row.label), str(row.source_video_id), spec


def make_video_level_balanced_sampler(dataset, hard_videos=None, hard_multiplier=1.75):
    hard_videos = set(hard_videos or [])
    labels = dataset.video_df["label"].to_numpy(dtype=np.int64)
    class_counts = np.bincount(labels, minlength=2).astype(np.float64)
    class_counts[class_counts == 0] = 1.0
    video_weights = np.asarray([1.0 / class_counts[label] for label in labels], dtype=np.float64)
    if hard_videos:
        video_ids = dataset.video_df["source_video_id"].astype(str).to_numpy()
        video_weights[np.isin(video_ids, list(hard_videos))] *= float(hard_multiplier)
    clip_weights = np.repeat(video_weights, dataset.clips_per_video)
    return WeightedRandomSampler(torch.DoubleTensor(clip_weights), num_samples=len(clip_weights), replacement=True)


def make_balanced_epoch_indices(dataset, hard_videos=None, hard_multiplier=1.75, rng=None):
    """Oversample video ids, then return all clip slots for each selected video.

    This preserves the video -> N clips framing while making class contribution per epoch
    approximately balanced. Use with torch.utils.data.SubsetRandomSampler or a DataLoader
    sampler argument.
    """
    rng = rng or np.random.default_rng(dataset.seed + dataset.epoch)
    hard_videos = set(hard_videos or [])
    labels = dataset.video_df["label"].to_numpy(dtype=np.int64)
    video_ids = dataset.video_df["source_video_id"].astype(str).to_numpy()
    class_to_indices = {label: np.flatnonzero(labels == label) for label in np.unique(labels)}
    target_count = max(len(indices) for indices in class_to_indices.values())
    selected_videos = []
    for label, indices in class_to_indices.items():
        weights = np.ones(len(indices), dtype=np.float64)
        if hard_videos:
            weights[np.isin(video_ids[indices], list(hard_videos))] *= float(hard_multiplier)
        weights = weights / weights.sum()
        replace = len(indices) < target_count
        selected = rng.choice(indices, size=target_count, replace=replace, p=weights)
        selected_videos.extend(selected.tolist())
    rng.shuffle(selected_videos)
    clip_indices = []
    for video_idx in selected_videos:
        base = int(video_idx) * dataset.clips_per_video
        clip_indices.extend(range(base, base + dataset.clips_per_video))
    return clip_indices


train_dataset = MultiClipVideoDataset(train_df, clips_per_video=CFG["clips_per_video_train"], train=True, seed=CFG["seed"])
val_dataset = MultiClipVideoDataset(
    val_df,
    clips_per_video=CFG["eval_clips_per_video"],
    train=False,
    deterministic=True,
    seed=CFG["seed"],
)
test_dataset = MultiClipVideoDataset(
    test_df,
    clips_per_video=CFG["eval_clips_per_video"],
    train=False,
    deterministic=True,
    seed=CFG["seed"],
)

train_sampler = make_video_level_balanced_sampler(train_dataset)
# Alternative exact balanced epoch:
# balanced_indices = make_balanced_epoch_indices(train_dataset)
# train_loader = DataLoader(train_dataset, batch_size=8, sampler=balanced_indices, num_workers=0, collate_fn=pad_collate_clips)
train_loader = DataLoader(train_dataset, batch_size=8, sampler=train_sampler, num_workers=0, collate_fn=pad_collate_clips)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, collate_fn=pad_collate_clips)

batch = next(iter(train_loader))
print("clip batch:", tuple(batch["clip"].shape))
print("labels:", batch["label"][:8].tolist())
print("lengths:", batch["length"][:8].tolist())
print("first specs:", batch["spec"][:3])


clip batch: (8, 3, 32, 224, 224)
labels: [0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0]
lengths: [24, 16, 16, 32, 16, 32, 32, 24]
first specs: [ClipSpec(video_id='R_non_hate_video_251', label=0, start=6, stride=3, length=24, segment=0, num_frames=24), ClipSpec(video_id='hate_video_152', label=1, start=2, stride=4, length=16, segment=0, num_frames=64), ClipSpec(video_id='R_non_hate_video_080', label=0, start=38, stride=1, length=16, segment=1, num_frames=64)]


## Multi-Clip Inference: Average Logits Per Video


In [24]:
@torch.no_grad()
def predict_video_logits(model, dataset, device="cuda", batch_size=8, num_workers=0):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=pad_collate_clips)
    model.eval()
    logits_by_video = defaultdict(list)
    labels_by_video = {}

    for batch in loader:
        clips = batch["clip"].to(device)
        labels = batch["label"].cpu().numpy().astype(int)
        logits = model(clips)
        if hasattr(logits, "logits"):
            logits = logits.logits
        logits = logits.detach().cpu()

        for video_id, label, logit in zip(batch["video_id"], labels, logits):
            logits_by_video[str(video_id)].append(logit)
            labels_by_video[str(video_id)] = int(label)

    rows = []
    for video_id, clip_logits in logits_by_video.items():
        mean_logits = torch.stack(clip_logits, dim=0).mean(dim=0)
        if mean_logits.numel() == 1:
            score = torch.sigmoid(mean_logits.flatten()[0]).item()
            logit_value = mean_logits.flatten()[0].item()
        else:
            score = torch.softmax(mean_logits, dim=0)[1].item()
            logit_value = mean_logits[1].item()
        rows.append(
            {
                "source_video_id": video_id,
                "label": labels_by_video[video_id],
                "mean_logit": logit_value,
                "prob_hate": float(score),
                "num_clips": len(clip_logits),
            }
        )
    return pd.DataFrame(rows)


def select_threshold_from_validation(val_pred_df):
    y_true = val_pred_df["label"].to_numpy(dtype=int)
    y_prob = val_pred_df["prob_hate"].to_numpy(dtype=float)
    thresholds = np.arange(0.10, 0.9001, 0.01)
    scores = [f1_score(y_true, y_prob >= threshold, zero_division=0) for threshold in thresholds]
    best_idx = int(np.argmax(scores))
    return float(thresholds[best_idx]), float(scores[best_idx])


def update_hard_videos_from_predictions(pred_df, threshold, low_conf_margin=0.10):
    pred = (pred_df["prob_hate"] >= threshold).astype(int)
    wrong = pred != pred_df["label"].astype(int)
    low_conf = (pred_df["prob_hate"] - threshold).abs() <= low_conf_margin
    return set(pred_df.loc[wrong | low_conf, "source_video_id"].astype(str))


## Training Loop Skeleton


In [25]:
def train_one_epoch(model, loader, optimizer, device="cuda", bce=True):
    model.train()
    total_loss = 0.0
    total = 0
    for batch in loader:
        clips = batch["clip"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(clips)
        if hasattr(logits, "logits"):
            logits = logits.logits
        if bce:
            logits = logits.flatten()
            loss = F.binary_cross_entropy_with_logits(logits, labels)
        else:
            loss = F.cross_entropy(logits, labels.long())
        loss.backward()
        optimizer.step()
        total_loss += float(loss.detach().cpu()) * labels.numel()
        total += labels.numel()
    return total_loss / max(1, total)


# Usage after each epoch:
# train_dataset.set_epoch(epoch)
# train_sampler = make_video_level_balanced_sampler(train_dataset, hard_videos=hard_videos)
# train_loader = DataLoader(train_dataset, batch_size=8, sampler=train_sampler, collate_fn=pad_collate_clips)
# train_loss = train_one_epoch(model, train_loader, optimizer, device=DEVICE)
#
# val_pred_df = predict_video_logits(model, val_dataset, device=DEVICE)
# threshold, val_f1 = select_threshold_from_validation(val_pred_df)
# hard_videos = update_hard_videos_from_predictions(val_pred_df, threshold)
#
# Final test:
# test_pred_df = predict_video_logits(model, test_dataset, device=DEVICE)
# test_pred_df["prediction"] = (test_pred_df["prob_hate"] >= threshold).astype(int)
